In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

GOLD_PATH = "src/de_etl_pipeline_loja/data/gold/vendas_produto_mensal"

def get_spark():
    builder = (
        SparkSession.builder
        .appName("Consulta_SQL_Gold")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config(
            "spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog",
        )
    )
    return configure_spark_with_delta_pip(builder).getOrCreate()

if __name__ == "__main__":
    spark = get_spark()

    df_gold = spark.read.format("delta").load(GOLD_PATH)
    df_gold.createOrReplaceTempView("vendas_produto_mensal")

    resultado = spark.sql("""
        SELECT ano, mes, produto, receita_total
        FROM vendas_produto_mensal
        WHERE ano = 2024
        ORDER BY mes, receita_total DESC
        LIMIT 10
    """)

    resultado.show()
    spark.stop()